# AI Sandbox: PDF -> Keyword Extraction -> DB Query Execution
This notebook demonstrates the pipeline using `litellm` and `instructor`.

In [1]:
import os
import fitz  # PyMuPDF
import instructor
from litellm import completion
from pydantic import BaseModel, Field
import sqlite3
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Ensure you have OPENROUTER_API_KEY set in your .env file


False

## 1. Define the Expected Output Schema
We use Pydantic to enforce the structure of the LLM's response. `instructor` uses this to constrain the model's output to strictly JSON.

In [2]:
class QueryExtraction(BaseModel):
    keywords: list[str] = Field(description="List of important keywords extracted from the attendance policy document.")
    sql_query: str = Field(description="SQLite query generated based on the document's content, targeting the FPI schema tables: policies, rules, employees, attendance_logs, point_history.")
    confidence_score: float = Field(description="Confidence score of the extraction between 0.0 and 1.0.")


## 2. PDF Extraction using PyMuPDF
We use `pymupdf` (fitz) to extract the full text from the SilverPeak Fabrication attendance policy PDF.

In [3]:
PDF_PATH = "/input_files/HR-811-A_SilverPeak_Fabrication_Attendance_Policy_Complex.pdf"

doc = fitz.open(PDF_PATH)
num_pages = len(doc)
pdf_text = ""
for page in doc:
    pdf_text += page.get_text()
doc.close()

print(f"Extracted {num_pages} pages, {len(pdf_text)} characters")
print("\n--- First 600 characters ---")
print(pdf_text[:600])


Extracted 2 pages, 1638 characters

--- First 600 characters ---
POLICY CODE: HR-811-A
Workforce Attendance Accountability Framework
Entity: SilverPeak Fabrication Ltd. | Region: Ontario, Canada
1. Policy Statement
SilverPeak Fabrication Ltd. maintains a structured attendance accountability system designed to
ensure operational continuity within a high-volume manufacturing environment. This policy establishes
a rolling accrual model with event-tier weighting, progressive intervention, and defined escalation
criteria.
2. Event Severity Classification
Tier
Event
Definition
Units Assigned
Tier I
Late Arrival (11–30 min)
Arrival beyond grace period
0.5
Tier I
E


## 3. Call the Model
Using litellm + instructor to get typed output. The system prompt includes the full FPI database schema so the model generates accurate SQLite queries.

In [ ]:
FPI_SCHEMA = """
Available SQLite tables (Fair Play Initiative - FPI schema):

  organizations (id TEXT PK, name TEXT, code TEXT, active BOOL)
  regions       (id TEXT PK, name TEXT, code TEXT, timezone TEXT, labor_laws TEXT)
  policies      (id TEXT PK, name TEXT, organization_id TEXT FK->organizations,
                 region_id TEXT FK->regions, active BOOL, effective_date DATE)
  rules         (id INTEGER PK AUTOINCREMENT, policy_id TEXT FK->policies,
                 name TEXT, condition TEXT CHECK(late|early|absence|no-call),
                 threshold INTEGER, points REAL, description TEXT, active BOOL)
  employees     (id INTEGER PK AUTOINCREMENT, first_name TEXT, last_name TEXT,
                 email TEXT UNIQUE, department TEXT, position TEXT, start_date DATE,
                 points REAL, trend TEXT, next_reset DATE,
                 organization_id TEXT FK->organizations,
                 region_id TEXT FK->regions, policy_id TEXT FK->policies)
  attendance_logs (id INTEGER PK, employee_id INTEGER FK->employees,
                   organization_id TEXT, region_id TEXT, policy_id TEXT, date DATE,
                   scheduled_in TEXT, scheduled_out TEXT, actual_in TEXT,
                   actual_out TEXT, violation TEXT, points REAL, status TEXT)
  point_history (id INTEGER PK, employee_id INTEGER FK->employees, date DATE,
                 type TEXT, points REAL,
                 status TEXT CHECK(Active|Excused|Approved), reason TEXT)
  alerts        (id INTEGER PK, organization_id TEXT, type TEXT, message TEXT,
                 created_at DATETIME)
"""

client = instructor.from_litellm(completion)

# Just make sure you have the API key in .env or os.environ for whichever provider you use.
# e.g. 'gpt-4o-mini', 'claude-3-haiku-20240307', 'ollama/llama3'
MODEL_NAME = "openrouter/openai/gpt-4o-mini"

try:
    extracted_data = client.chat.completions.create(
        model=MODEL_NAME,
        response_model=QueryExtraction,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data extraction and SQL generation assistant for an HR attendance "
                    "management system called the Fair Play Initiative (FPI). "
                    "Extract important keywords from attendance policy documents and generate a "
                    "meaningful SQLite query using the schema below. "
                    "Queries should help answer a relevant HR question (e.g. 'list all violation rules "
                    "and their point values', 'find employees at risk based on thresholds'). "
                    "Only output structured data.\n\n"
                    f"{FPI_SCHEMA}"
                )
            },
            {
                "role": "user",
                "content": (
                    "Extract keywords and generate a relevant SQL query for the following attendance "
                    f"policy document:\n\n{pdf_text[:4000]}"
                )
            }
        ]
    )

    print("--- Extraction Results ---")
    print(f"Keywords: {extracted_data.keywords}")
    print(f"\nSQL Query:\n{extracted_data.sql_query}")
    print(f"\nConfidence: {extracted_data.confidence_score}")
except Exception as e:
    print(f"Error calling model: {e}")
    extracted_data = None


--- Extraction Results ---
Keywords: ['attendance', 'accountability', 'event-tier weighting', 'late arrival', 'early departure', 'full shift absence', 'partial absence', 'no call no show', 'rolling accrual', 'escalation criteria', 'coaching conference', 'formal warning', 'final warning', 'termination review', 'protected leave exclusions']

SQL Query:
SELECT r.name AS rule_name, r.points AS point_value FROM rules r JOIN policies p ON r.policy_id = p.id WHERE p.name = 'Workforce Attendance Accountability Framework' AND p.organization_id = (SELECT id FROM organizations WHERE name = 'SilverPeak Fabrication Ltd.') AND p.region_id = (SELECT id FROM regions WHERE name = 'Ontario, Canada') AND r.active = 1;

Confidence: 0.95


## 4. Execute Query on FPI Schema Database
Set up a local in-memory SQLite DB mirroring the FPI schema, seeded with SilverPeak policy data, to verify the AI-generated SQL query works.

In [5]:
if extracted_data:
    conn = sqlite3.connect(':memory:')
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # --- FPI Schema ---
    cursor.executescript("""
        CREATE TABLE organizations (
            id   TEXT PRIMARY KEY,
            name TEXT,
            code TEXT,
            active BOOLEAN DEFAULT 1
        );

        CREATE TABLE regions (
            id         TEXT PRIMARY KEY,
            name       TEXT,
            code       TEXT,
            timezone   TEXT,
            labor_laws TEXT
        );

        CREATE TABLE policies (
            id              TEXT PRIMARY KEY,
            name            TEXT,
            organization_id TEXT REFERENCES organizations(id),
            region_id       TEXT REFERENCES regions(id),
            active          BOOLEAN DEFAULT 1,
            effective_date  DATE
        );

        CREATE TABLE rules (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            policy_id   TEXT REFERENCES policies(id),
            name        TEXT,
            condition   TEXT,
            threshold   INTEGER,
            points      REAL,
            description TEXT,
            active      BOOLEAN DEFAULT 1
        );

        CREATE TABLE employees (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name      TEXT,
            last_name       TEXT,
            email           TEXT UNIQUE,
            department      TEXT,
            position        TEXT,
            start_date      DATE,
            points          REAL DEFAULT 0,
            trend           TEXT DEFAULT 'stable',
            next_reset      DATE,
            organization_id TEXT REFERENCES organizations(id),
            region_id       TEXT REFERENCES regions(id),
            policy_id       TEXT REFERENCES policies(id)
        );

        CREATE TABLE attendance_logs (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            employee_id     INTEGER REFERENCES employees(id),
            organization_id TEXT,
            region_id       TEXT,
            policy_id       TEXT,
            date            DATE,
            scheduled_in    TEXT,
            scheduled_out   TEXT,
            actual_in       TEXT,
            actual_out      TEXT,
            violation       TEXT,
            points          REAL DEFAULT 0,
            status          TEXT DEFAULT 'Compliant'
        );

        CREATE TABLE point_history (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            employee_id INTEGER REFERENCES employees(id),
            date        DATE,
            type        TEXT,
            points      REAL,
            status      TEXT DEFAULT 'Active',
            reason      TEXT
        );

        CREATE TABLE alerts (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            organization_id TEXT,
            type            TEXT,
            message         TEXT,
            created_at      DATETIME DEFAULT CURRENT_TIMESTAMP
        );
    """)

    # --- Seed Data: SilverPeak Fabrication ---
    cursor.execute(
        "INSERT INTO organizations VALUES ('silverpeak', 'SilverPeak Fabrication', 'SLVPK', 1)"
    )
    cursor.execute(
        "INSERT INTO regions VALUES ('us-mw', 'US Midwest', 'US-MW', 'America/Chicago', 'Illinois Labor Code')"
    )
    cursor.execute(
        "INSERT INTO policies VALUES ('silverpeak-pol', 'SilverPeak Attendance Policy', 'silverpeak', 'us-mw', 1, '2024-01-01')"
    )

    cursor.executemany(
        "INSERT INTO rules (policy_id, name, condition, threshold, points, description, active) VALUES (?,?,?,?,?,?,?)",
        [
            ('silverpeak-pol', 'Tardiness (>7 min)',   'late',    7,  0.5, 'Arriving more than 7 minutes after scheduled start',  1),
            ('silverpeak-pol', 'Tardiness (>30 min)',  'late',   30,  1.0, 'Arriving more than 30 minutes after scheduled start', 1),
            ('silverpeak-pol', 'Early Departure',      'early',  15,  0.5, 'Leaving more than 15 minutes before scheduled end',   1),
            ('silverpeak-pol', 'Unexcused Absence',    'absence', 0,  1.5, 'Full-day absence with prior notification',            1),
            ('silverpeak-pol', 'No-Call No-Show',      'no-call', 0,  3.0, 'Full-day absence without any notification',           1),
        ]
    )

    cursor.executemany(
        "INSERT INTO employees (first_name, last_name, email, department, position, start_date, points, trend, next_reset, organization_id, region_id, policy_id) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
        [
            ('Alice', 'Chen',    'achen@silverpeak.com',   'Fabrication', 'Machinist',       '2021-03-15', 3.5, 'up',     '2026-06-01', 'silverpeak', 'us-mw', 'silverpeak-pol'),
            ('Bob',   'Torres',  'btorres@silverpeak.com', 'Fabrication', 'Welder',          '2019-07-01', 1.0, 'stable', '2026-06-01', 'silverpeak', 'us-mw', 'silverpeak-pol'),
            ('Carol', 'Kim',     'ckim@silverpeak.com',    'Quality',     'QA Inspector',    '2022-11-10', 8.5, 'up',     '2026-06-01', 'silverpeak', 'us-mw', 'silverpeak-pol'),
            ('David', 'Patel',   'dpatel@silverpeak.com',  'Shipping',    'Logistics Lead',  '2020-02-28', 0.5, 'down',   '2026-06-01', 'silverpeak', 'us-mw', 'silverpeak-pol'),
        ]
    )

    conn.commit()

    # --- Execute AI-generated query ---
    try:
        print(f"Executing:\n{extracted_data.sql_query}\n")
        cursor.execute(extracted_data.sql_query)
        rows = cursor.fetchall()
        col_names = [d[0] for d in cursor.description] if cursor.description else []
        print(f"Columns : {col_names}")
        print("\n--- Query Results ---")
        for row in rows:
            print(dict(row))
    except Exception as e:
        print(f"\nSQL Error: {e}")


Executing:
SELECT r.name AS rule_name, r.points AS point_value FROM rules r JOIN policies p ON r.policy_id = p.id WHERE p.name = 'Workforce Attendance Accountability Framework' AND p.organization_id = (SELECT id FROM organizations WHERE name = 'SilverPeak Fabrication Ltd.') AND p.region_id = (SELECT id FROM regions WHERE name = 'Ontario, Canada') AND r.active = 1;

Columns : ['rule_name', 'point_value']

--- Query Results ---


## 5. SQL Explorer
Edit the `sql` variable and re-run this cell to query the in-memory DB.
The connection (`conn`) was created in the cell above and stays open for the lifetime of the kernel.

In [ ]:
def run_query(sql):
    cur = conn.cursor()
    try:
        cur.execute(sql)
        rows = cur.fetchall()
        col_names = [d[0] for d in cur.description] if cur.description else []
        print(' | '.join(col_names))
        print('-' * 60)
        for row in rows:
            print(' | '.join(str(v) for v in tuple(row)))
        print()
        print(str(len(rows)) + ' row(s)')
    except Exception as e:
        print('SQL Error: ' + str(e))


# --- Change the query below and re-run this cell ---
sql = """
SELECT name, condition, threshold, points, description
FROM   rules
WHERE  policy_id = 'silverpeak-pol'
ORDER  BY points DESC
"""

# Other example queries (uncomment to try):
# sql = "SELECT * FROM employees ORDER BY points DESC"
# sql = "SELECT * FROM policies"
# sql = "SELECT * FROM organizations"
# sql = "SELECT e.first_name, e.last_name, e.points, r.name AS region FROM employees e JOIN regions r ON e.region_id = r.id"
# sql = "SELECT name FROM sqlite_master WHERE type='table'"

run_query(sql)


name | condition | threshold | points | description
------------------------------------------------------------
No-Call No-Show | no-call | 0 | 3.0 | Full-day absence without any notification
Unexcused Absence | absence | 0 | 1.5 | Full-day absence with prior notification
Tardiness (>30 min) | late | 30 | 1.0 | Arriving more than 30 minutes after scheduled start
Tardiness (>7 min) | late | 7 | 0.5 | Arriving more than 7 minutes after scheduled start
Early Departure | early | 15 | 0.5 | Leaving more than 15 minutes before scheduled end

5 row(s)


In [7]:
sql = "SELECT * FROM employees ORDER BY points DESC"
run_query(sql)

id | first_name | last_name | email | department | position | start_date | points | trend | next_reset | organization_id | region_id | policy_id
------------------------------------------------------------
3 | Carol | Kim | ckim@silverpeak.com | Quality | QA Inspector | 2022-11-10 | 8.5 | up | 2026-06-01 | silverpeak | us-mw | silverpeak-pol
1 | Alice | Chen | achen@silverpeak.com | Fabrication | Machinist | 2021-03-15 | 3.5 | up | 2026-06-01 | silverpeak | us-mw | silverpeak-pol
2 | Bob | Torres | btorres@silverpeak.com | Fabrication | Welder | 2019-07-01 | 1.0 | stable | 2026-06-01 | silverpeak | us-mw | silverpeak-pol
4 | David | Patel | dpatel@silverpeak.com | Shipping | Logistics Lead | 2020-02-28 | 0.5 | down | 2026-06-01 | silverpeak | us-mw | silverpeak-pol

4 row(s)
